<h2>Impact (v0.5) - 'Impact (v0.5).ipynb'</h2>

The aim of the 'Impact' notebooks are to address the following questions laid out in Q1.1.4 of 'Working Paper 1' focusing on 'Task 1.1 - GFI Trajectories and Relevance to African Shipping'.

<u><h4>How do the fleet-level transport cost outputs generated in '3. LEAP III Inventories (v0.1).ipynb' map to commodity flow volumes and GDP-level welfare effects?</h4></u>

We will first focus on the mapping to commodity flows, for which we will need seaborne trade portfolios as a first step. Once we have these, we can then proceed with the mapping step between fleet-level transport cost outputs and commodity flows. To achieve this, we will first integrate Sea Distances derived in the 'CERDI Sea Distances' table. We can then derive the product of the sea distance with the cargo volume, and multiply this by the transport cost intensities identified in the matching '3. LEAP III Inventories (v0.1).ipynb' notebook.

<u><h4> How sensitive are these estimates to data availability for African trade routes where vessel-tracking data is incomplete?</h4></u>

The seaborne trade portfolios are here derived from UNCTAD's Trade-and-Transport database. This primarily owes to the fact that the database is able to provide bilateral seaborne trade portfolios 'off-the-shelf' to users, however there are limitations with the data within the dataset that principally owe to the quality of input Comtrade data utilised for the year in question. Care should therefore taken to explore and understand the quality associated with the Trade-and-Transport-derived seaborne trade portfolio derived.

<u><h4>Version Documentation and Future Work</h4></u>

'v0.2' cleaned up the analysis code by introducing the cost intensity estimation functions that were previously in the 'inventories' notebooks. For example, the 'get_usd_per_tnm' function is included here and applied to voyages identified as relevant to each individual trade record. The NZF costs associated with each trade record are now evaluated like this. 'v0.3' introduced the exporter- and importer-side trade portfolios at an HS-digit 4 level of resolution. With this implemented, the matching algorithm is now be being evaluated using the cargo weight field rather than value, and the process will be being conducted at the HS-digit 4 level of resolution. Since we are now confident regarding the use of the Global Shipping Watch dataset for the purposes of matching HS codes with facilitating vessel types, we should then be in good shape to take the analysis forward using the results produced in this notebook. 'v0.3' also subsumed the 'inventories' notebooks. In 'v0.4' sought to include transport costs data from UNCTAD's Trade-and-Transport database. However, there are now concerns regarding this dataset as it is importer-side only, and therefore has odd looking exporter-side portfolios.

Instead, it makes sense for us to utilise trade portfolios derived from UN Comtrade, where these are submitted with disaggregation by Transport Mode. Looking at the 16 countries of interest across African and Caribbean economies we're interested in the summer of 2026, 13 of these have a trade portfolio with MoT Breakdown available across at least one year within the last decade. Therefore this approach makes more sense, and 'v0.5' is amended to work with Comtrade import and export trade portfolios downloaded from the platform.

Limitations of the method include the following:

<ol type=1>
    <li>There are inherent limitations when using the Trade-and-Transport dataset, further details of which can be found its underlying documentation.</li>
    <li>2018 data is used, which is increasingly out of date and may not reflect the latest changes impacting each of the country's of interest.</li>
</ol>

In [2]:
# Import packages
import os
import ast
import base64
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 250)
pd.set_option('display.max_rows', 250)
pd.options.mode.chained_assignment = None
from IPython.display import HTML
import matplotlib.pyplot as plt
import matplotlib.style
plt.style.use('seaborn-v0_8')

def voyages_analysis(voys):
    # Define a function to analyses key statistics from a given set of voyages, 'voys'.
    print("Number of Voyages found: {0}".format(voys.shape[0]))
    print("Total size of the fleet: {0} vessels".format(len(voys.imo.unique())))
    print("{0} vessel types contained in the dataset: {1}".format(len(voys.vessel_type.unique()), voys.vessel_type.unique()))
    print("Total transport work capacity: {0} million deadweight-tonne nautical miles".format(np.round(voys.dwtnm.sum() / 1e6, 1)))
    print("Total cargo transported: {0} tonnes".format(np.round(voys.tonnes.sum(), 1)))
    print("Total transport work: {0} million tonne nautical miles".format(np.round(voys.tnm.sum() / 1e6, 1)))
    print("Average build year of the fleet: {0}".format(int(np.round(voys.year_of_built.mean(), 0))))
    print("Total Energy Demands: {0} EJ".format(np.round(voys.energy_tj.sum(), 1)))
    print("Total CO2 produced: {0} tonnes".format(np.round(voys.co2_t.sum(), 1)))

def get_usd_per_dwtnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carrying capacity) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values * voys.deadweight.values).sum() / voys.deadweight.sum()
    
def get_usd_per_tnm(voys, field_name):
    # Derive the deadweight-average of the cost intensity (per unit of cargo-carried) for the field 'field_name' in an input selection of voyages, 'voys'.
    return (voys[field_name].values / voys.class_util.values * voys.tonnes.values).sum() / voys.tonnes.sum()

def download_link(df, title = "Download CSV file", filename = "data.csv"):
    csv = df.to_csv()
    b64 = base64.b64encode(csv.encode())
    payload = b64.decode()
    html = '<a download="{filename}" href="data:text/csv;base64,{payload}" target="_blank">{title}</a>'
    html = html.format(payload=payload,title=title,filename=filename)
    return HTML(html)

# Define functions for plotting a count of values in a given data field.
int_arr_voys_field_count = lambda field: int_arr_voys[field].value_counts().to_frame().reset_index()
int_dep_voys_field_count = lambda field: int_dep_voys[field].value_counts().to_frame().reset_index()

# define function for counting values in each data field
trade_im_field_count = lambda field: trade_im[field].value_counts().to_frame().reset_index()
trade_ex_field_count = lambda field: trade_ex[field].value_counts().to_frame().reset_index()

version = "v0.5"
input_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/"
output_dir = "/Users/apple/repos/PubEnv/2026_nzf_impact/impact/"

# 1. Set up the notebook

## 1.1 Voyages with FEM Results

In [3]:
voys_fem_d = {
    "source_iso":str, "source_iso_code":str, "source_iso_3":str, "source_iso_2":str, 
    "dest_iso_code":str, "dest_iso_code":str, "dest_iso_3":str, "dest_iso_2":str, 
    "source_region_code":str, "source_sub_region_code":str, "dest_region_code":str, "dest_sub_region_code":str, 
    "source_ldc":str, "source_sids":str, "dest_ldc":str, "dest_sids":str
}

voys_fem = pd.read_csv("/Users/apple/repos/datasets/ucl/fleet evolution model/2026 annex vi amendments/2. NZF Impact Costs for 2018 IMO IV Voyages/voys_fem_0.1.csv", dtype=voys_fem_d)
voys_fem["dwtnm"] = voys_fem.deadweight * voys_fem.dist_travelled_ais
voys_fem["tonnes"] = voys_fem.deadweight * voys_fem.class_util
voys_fem["tnm"] = voys_fem.tonnes * voys_fem.dist_travelled_ais
voys_fem.loc[voys_fem.source_country == "Namibia", "source_iso_2"] = "NA"
voys_fem.loc[voys_fem.dest_country == "Namibia", "dest_iso_2"] = "NA"

countries_set = pd.Series(list(set(list(voys_fem.source_country.unique()) + list(voys_fem.dest_country.unique())))).sort_values()
print("Number of countries present in dataset: {0}.".format(len(countries_set)))
print("Number of IMO numbers present in dataset: {0}.".format(len(voys_fem.imo.unique())))
print("Total CO2 contained within the dataset: {0} million tonnes.".format(np.round(voys_fem.co2_t.sum() / 1e6, 1)))
print("Vessel types present in dataset: \n\n{0}\n".format(pd.Series(voys_fem.vessel_type.unique()).sort_values().values))
#print("\nCountries in dataset: \n\n", countries_set.values, "\n")
voys_fem.head(2)

Number of countries present in dataset: 207.
Number of IMO numbers present in dataset: 26132.
Total CO2 contained within the dataset: 544.7 million tonnes.
Vessel types present in dataset: 

['Bulk carrier' 'Chemical tanker' 'Container' 'General cargo'
 'Liquefied gas tanker' 'Oil tanker' 'Refrigerated bulk' 'Vehicle']



,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 1.2 Let's set the country we're interested in and set up a results folder.

In [4]:
# Select one of our countries to work on...
iso_code = "516"
iso_country = "Namibia"
year = 2024
gdp = 13.460e9

# Create output Journey directory folder if necessary
os.makedirs(output_dir+"{0}/".format(iso_country)) if not os.path.exists(output_dir+"{0}/".format(iso_country)) else "Output directory already exists."

'Output directory already exists.'

# 2. Let's consider Imports

## 2.1 Read-in Trade dataset

In [5]:
trade_im_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_im = pd.read_csv(input_dir+"trade/{0}/trade_im_dist_v0.6.csv".format(iso_country), dtype=trade_im_d)
trade_im['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_im.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_im.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.netWgt.sum() / 1e6, 1)))
trade_im.head(2)

Aggregate trade portfolio statistics for 'Namibia': 

Number of trade records present in dataset: 7587
Number of partner countries present in dataset: 130
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$3541.5m
Total weight within the dataset: 2955.7 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,516,Namibia,044,Bahamas,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,71.0,71.0,1159.641,1008.385,1159.641,Meat of bovine anima,0201 | Meat of bovine anima,Bahamas | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Bahamas | 02 | Meat and edible meat,[Container],11146.0,6925.801166,491.731883
1,1,516,Namibia,392,Japan,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,14.4,14.4,162.514,125.815,162.514,Meat of bovine anima,0201 | Meat of bovine anima,Japan | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Japan | 02 | Meat and edible meat,[Container],16977.0,10549.015467,151.905823


## 2.2 Derive Internationally Arriving Voyages

In [6]:
# International Arrivals
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_arr_voys)
int_arr_voys.head(2)

Number of Voyages found: 603
Total size of the fleet: 321 vessels
8 vessel types contained in the dataset: ['Refrigerated bulk' 'General cargo' 'Container' 'Bulk carrier'
 'Chemical tanker' 'Vehicle' 'Oil tanker' 'Liquefied gas tanker']
Total transport work capacity: 79919.6 million deadweight-tonne nautical miles
Total cargo transported: 12986311.8 tonnes
Total transport work: 48153.3 million tonne nautical miles
Average build year of the fleet: 2005
Total Energy Demands: 8616.0 EJ
Total CO2 produced: 632850.7 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 2.3 Evaluate NZF Costs

In [15]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_im.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_arr_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_arr_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_arr_voys_vess.shape[0] == 0:
                int_arr_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_arr_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_arr_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]
            
            if int_arr_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
                continue
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_arr_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_im["bau_usd_"+year] = bau_usd
    trade_im["levy_usd_"+year] = levy_usd
    trade_im["lh_levy_usd_"+year] = lh_levy_usd
    trade_im["asis_usd_"+year] = asis_usd
    trade_im["brazil_usd_"+year] = brazil_usd
    trade_im["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_im.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_im.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_im.cifvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_im.netWgt.sum() / 1e6, 1)))


print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.bau_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_im.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.lh_levy_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_im.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.asis_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.brazil_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_im.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_im.liscr_usd_41.sum() / trade_im.cifvalue.sum(), 1)
))

trade_im_na = trade_im[(trade_im[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_im.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(trade_im_na.primaryValue.sum(), round(100 * trade_im_na.primaryValue.sum() / trade_im.primaryValue.sum(), 1)))

trade_im = trade_im[(~trade_im[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (~trade_im.fac_vessel_types.str.len() == 0)]
trade_im.to_csv(output_dir + "{0}/impact_im_{1}.csv".format(iso_country, version), index=False)
trade_im.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['3827 | Mixtures containing ' '8807 | Aircraft; parts of g'
 '9620 | Monopods, bipods, tr']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 7587
Number of partner countries present in dataset: 130

Total value of trade: US$3541.5m
Total weight within the dataset: 2955.7 million kilograms


Total FEM costs in 2041...

BAU: US$136.0m (3.8% ad-val)
Levy: US$273.6m (7.7% ad-val)
Levy (low-then-high): US$269.6m (7.6% ad-val)
'as-is': US$258.7m (7.3% ad-val)
Brazil: US$268.4m (7.6% ad-val)
LISCR: US$186.2m (5.3% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,516,Namibia,044,Bahamas,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,71.0,71.0,1159.641,1008.385,1159.641,Meat of bovine anima,0201 | Meat of bovine anima,Bahamas | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Bahamas | 02 | Meat and edible meat,[Container],11146.0,6925.801166,491.731883,3.335018,3.320812,3.321279,3.321748,3.321748,3.331377,3.504841,6.684371,4.084436,3.919065,3.550637,3.565115,3.424089,6.311238,4.478112,4.040698,3.705525,3.625933,3.35336,6.954384,6.905101,5.077614,4.403759,4.076178,3.283491,7.401245,6.841589,5.882953,5.541750,4.310853,3.214890,7.508171,7.343757,6.978072,7.083090,4.713070,3.144995,7.389331,7.486707,7.171406,7.566115,5.527987,3.079277,6.598778,6.924917,6.710491,7.032673,5.353050,3.012521,5.891669,6.013700,6.066009,7.154246,5.835547
1,1,516,Namibia,392,Japan,000,World,Import,Sea,Meat of bovine animals; fresh or chilled,0201,kg,14.4,14.4,162.514,125.815,162.514,Meat of bovine anima,0201 | Meat of bovine anima,Japan | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Japan | 02 | Meat and edible meat,[Container],16977.0,10549.015467,151.905823,1.030254,1.025865,1.026010,1.026154,1.026154,1.029129,1.082715,2.064936,1.261764,1.210678,1.096863,1.101335,1.057770,1.949668,1.383378,1.248253,1.144711,1.120123,1.03592,2.148348,2.133124,1.568577,1.360409,1.259213,1.014336,2.286393,2.113504,1.817362,1.711958,1.331709,0.993144,2.319424,2.268633,2.155666,2.188108,1.455962,0.971552,2.282712,2.312794,2.215391,2.337325,1.707706,0.951250,2.038495,2.139245,2.073005,2.172533,1.653664,0.930628,1.820054,1.857752,1.873912,2.210090,1.802717


## 2.4 Inpsect Output

### 2.4.1 Total Costs per Scenario

In [8]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_im["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_im["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_im["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_im["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,144.7,142.3,140.2,138.1,136.0,133.9,131.9,129.9
1,Levy,235.3,227.2,250.9,274.7,273.6,264.6,238.7,222.4
2,Levy (LH),160.4,175.3,247.7,260.0,269.6,265.1,247.6,223.9
3,'as-is',154.2,162.5,189.5,221.7,258.7,258.0,244.6,226.9
4,Brazil,144.7,151.0,175.5,209.6,268.4,271.0,266.7,261.7
5,LISCR,145.1,148.7,165.5,172.7,186.2,215.9,212.4,219.4



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,4.1,4.0,4.0,3.9,3.8,3.8,3.7,3.7
1,Levy,6.6,6.4,7.1,7.8,7.7,7.5,6.7,6.3
2,Levy (LH),4.5,5.0,7.0,7.3,7.6,7.5,7.0,6.3
3,'as-is',4.4,4.6,5.4,6.3,7.3,7.3,6.9,6.4
4,Brazil,4.1,4.3,5.0,5.9,7.6,7.7,7.5,7.4
5,LISCR,4.1,4.2,4.7,4.9,5.3,6.1,6.0,6.2



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,1.075,1.057,1.041,1.026,1.010,0.995,0.980,0.965
1,Levy,1.748,1.688,1.864,2.041,2.033,1.966,1.773,1.652
2,Levy (LH),1.192,1.303,1.841,1.931,2.003,1.970,1.839,1.663
3,'as-is',1.146,1.207,1.408,1.647,1.922,1.917,1.818,1.686
4,Brazil,1.075,1.122,1.304,1.557,1.994,2.013,1.981,1.945
5,LISCR,1.078,1.105,1.230,1.283,1.383,1.604,1.578,1.630


### 2.4.2 Costs per Scenario relative to BAU

In [9]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / trade_im.cifvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_im["levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["lh_levy_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["asis_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["brazil_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_im["liscr_usd_{0}".format(year)].sum() - trade_im["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_im_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,90.6,84.9,110.8,136.7,137.6,130.8,106.8,92.5
1,Levy (LH),15.7,33.0,107.6,121.9,133.6,131.2,115.7,94.0
2,'as-is',9.6,20.2,49.4,83.6,122.7,124.1,112.8,97.1
3,Brazil,-0.0,8.7,35.4,71.5,132.4,137.1,134.8,131.9
4,LISCR,0.4,6.4,25.4,34.7,50.2,82.1,80.5,89.5



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,2.6,2.4,3.1,3.9,3.9,3.7,3.0,2.6
1,Levy (LH),0.4,0.9,3.0,3.4,3.8,3.7,3.3,2.7
2,'as-is',0.3,0.6,1.4,2.4,3.5,3.5,3.2,2.7
3,Brazil,-0.0,0.2,1.0,2.0,3.7,3.9,3.8,3.7
4,LISCR,0.0,0.2,0.7,1.0,1.4,2.3,2.3,2.5



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.673,0.631,0.823,1.015,1.022,0.972,0.794,0.688
1,Levy (LH),0.117,0.245,0.799,0.906,0.993,0.975,0.859,0.698
2,'as-is',0.071,0.150,0.367,0.621,0.912,0.922,0.838,0.721
3,Brazil,-0.000,0.065,0.263,0.531,0.984,1.019,1.002,0.980
4,LISCR,0.003,0.048,0.189,0.258,0.373,0.610,0.598,0.665


# 3. Let's consider Exports

## 3.1 Read-in Trade dataset

In [10]:
trade_ex_d = {"reporterCode":"str", "partnerCode":"str", "partner2Code":"str", "cmdCode":"str", "HS2":"str"}
trade_ex = pd.read_csv(input_dir+"trade/{0}/trade_ex_dist_v0.6.csv".format(iso_country), dtype=trade_ex_d)
trade_ex['fac_vessel_types'] = [ast.literal_eval(x) for x in trade_ex.fac_vessel_types.values]

print("Aggregate trade portfolio statistics for '{0}': \n".format(iso_country))
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("Modes of transport present in dataset: {0}".format(trade_ex.motDesc.unique()))
print("Total value within the dataset: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.netWgt.sum() / 1e6, 1)))
trade_ex.head(2)

Aggregate trade portfolio statistics for 'Namibia': 

Number of trade records present in dataset: 3552
Number of partner countries present in dataset: 114
Modes of transport present in dataset: ['Sea']
Total value within the dataset: US$2931.7m
Total weight within the dataset: 1866.6 million kilograms


,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm
0,0,516,Namibia,056,Belgium,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,72363.0,72363.0,772543.778,772543.778,772543.778,Meat of bovine anima,0201 | Meat of bovine anima,Belgium | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Belgium | 02 | Meat and edible meat,[Container],10468.0,6504.511628,470685.974937
1,1,516,Namibia,180,Dem. Rep. of the Congo,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,55.0,55.0,217.547,217.547,217.547,Meat of bovine anima,0201 | Meat of bovine anima,Dem. Rep. of the Congo | 0201 | Meat of bovine...,02,02 | Meat and edible meat,Meat and edible meat,Dem. Rep. of the Congo | 02 | Meat and edible ...,[Container],2222.0,1380.686362,75.937750


## 3.2 Derive Internationally Departing Voyages

In [11]:
# International Departures
int_arr_voys = voys_fem[(voys_fem.dest_country == iso_country) & (voys_fem.source_country != iso_country)].reset_index(drop=True)
int_arr_voys.to_csv(output_dir + "{0}/int_arr_voys_{1}.csv".format(iso_country, version), index=False)
int_dep_voys = voys_fem[(voys_fem.source_country == iso_country) & (voys_fem.dest_country != iso_country)].reset_index(drop=True)
int_dep_voys.to_csv(output_dir + "{0}/int_dep_voys_{1}.csv".format(iso_country, version), index=False)
voyages_analysis(int_dep_voys)
int_dep_voys.head(2)

Number of Voyages found: 603
Total size of the fleet: 320 vessels
8 vessel types contained in the dataset: ['Refrigerated bulk' 'General cargo' 'Container' 'Bulk carrier'
 'Chemical tanker' 'Vehicle' 'Oil tanker' 'Liquefied gas tanker']
Total transport work capacity: 59400.3 million deadweight-tonne nautical miles
Total cargo transported: 13009049.2 tonnes
Total transport work: 35763.5 million tonne nautical miles
Average build year of the fleet: 2005
Total Energy Demands: 5984.1 EJ
Total CO2 produced: 450567.1 tonnes


,voy_idx,imo,mmsi,source_iso,source_iso_code,source_iso_3,source_iso_2,source_country,source_name,dest_iso,dest_iso_code,dest_iso_3,dest_iso_2,dest_country,dest_name,start_date,end_date,duration_hrs_voy,duration_hrs_stop,duration_hrs,dist_travelled_ais,foc_hfo_t_voy,foc_hfo_t_stop,foc_hfo_t,foc_mdo_t_voy,foc_mdo_t_stop,foc_mdo_t,foc_lng_t_voy,foc_lng_t_stop,foc_lng_t,foc_methanol_t_voy,foc_methanol_t_stop,foc_methanol_t,energy_tj_voy,energy_tj_stop,co2_t_voy,co2_t_stop,co2_t,source_region_code,source_region,source_sub_region_code,source_sub_region,source_income_wesp,source_ldc,source_sids,dest_region_code,dest_region,dest_sub_region_code,dest_sub_region,dest_income_wesp,dest_ldc,dest_sids,energy_tj,size_id,vessel_type,size_range,capacity_unit,flag,year_of_built,gross_tonnage,deadweight,teu,gas_cap,passengers,cars,engine_type,me_fuel,class_med_aer,class_util,_voys_vess_m,_fem_m,bau_bio35_usd_per_dwtnm_26,bau_bio35_usd_per_dwtnm_29,bau_bio35_usd_per_dwtnm_32,bau_bio35_usd_per_dwtnm_35,bau_bio35_usd_per_dwtnm_38,bau_bio35_usd_per_dwtnm_41,bau_bio35_usd_per_dwtnm_44,bau_bio35_usd_per_dwtnm_47,bau_bio35_usd_per_dwtnm_50,levy_bio35_usd_per_dwtnm_26,levy_bio35_usd_per_dwtnm_29,levy_bio35_usd_per_dwtnm_32,levy_bio35_usd_per_dwtnm_35,levy_bio35_usd_per_dwtnm_38,levy_bio35_usd_per_dwtnm_41,levy_bio35_usd_per_dwtnm_44,levy_bio35_usd_per_dwtnm_47,levy_bio35_usd_per_dwtnm_50,lh_levy_bio35_usd_per_dwtnm_26,lh_levy_bio35_usd_per_dwtnm_29,lh_levy_bio35_usd_per_dwtnm_32,lh_levy_bio35_usd_per_dwtnm_35,lh_levy_bio35_usd_per_dwtnm_38,lh_levy_bio35_usd_per_dwtnm_41,lh_levy_bio35_usd_per_dwtnm_44,lh_levy_bio35_usd_per_dwtnm_47,lh_levy_bio35_usd_per_dwtnm_50,asis_bio35_usd_per_dwtnm_26,asis_bio35_usd_per_dwtnm_29,asis_bio35_usd_per_dwtnm_32,asis_bio35_usd_per_dwtnm_35,asis_bio35_usd_per_dwtnm_38,asis_bio35_usd_per_dwtnm_41,asis_bio35_usd_per_dwtnm_44,asis_bio35_usd_per_dwtnm_47,asis_bio35_usd_per_dwtnm_50,brazil_bio35_usd_per_dwtnm_26,brazil_bio35_usd_per_dwtnm_29,brazil_bio35_usd_per_dwtnm_32,brazil_bio35_usd_per_dwtnm_35,brazil_bio35_usd_per_dwtnm_38,brazil_bio35_usd_per_dwtnm_41,brazil_bio35_usd_per_dwtnm_44,brazil_bio35_usd_per_dwtnm_47,brazil_bio35_usd_per_dwtnm_50,liscr_bio35_usd_per_dwtnm_26,liscr_bio35_usd_per_dwtnm_29,liscr_bio35_usd_per_dwtnm_32,liscr_bio35_usd_per_dwtnm_35,liscr_bio35_usd_per_dwtnm_38,liscr_bio35_usd_per_dwtnm_41,liscr_bio35_usd_per_dwtnm_44,liscr_bio35_usd_per_dwtnm_47,liscr_bio35_usd_per_dwtnm_50,liscr_tolng_bio35_usd_per_dwtnm_26,liscr_tolng_bio35_usd_per_dwtnm_29,liscr_tolng_bio35_usd_per_dwtnm_32,liscr_tolng_bio35_usd_per_dwtnm_35,liscr_tolng_bio35_usd_per_dwtnm_38,liscr_tolng_bio35_usd_per_dwtnm_41,liscr_tolng_bio35_usd_per_dwtnm_44,liscr_tolng_bio35_usd_per_dwtnm_47,liscr_tolng_bio35_usd_per_dwtnm_50,_bau_temp_m,dwtnm,bau_bio35_usd_26,bau_bio35_usd_29,bau_bio35_usd_32,bau_bio35_usd_35,bau_bio35_usd_38,bau_bio35_usd_41,bau_bio35_usd_44,bau_bio35_usd_47,bau_bio35_usd_50,levy_bio35_usd_26,levy_bio35_usd_29,levy_bio35_usd_32,levy_bio35_usd_35,levy_bio35_usd_38,levy_bio35_usd_41,levy_bio35_usd_44,levy_bio35_usd_47,levy_bio35_usd_50,lh_levy_bio35_usd_26,lh_levy_bio35_usd_29,lh_levy_bio35_usd_32,lh_levy_bio35_usd_35,lh_levy_bio35_usd_38,lh_levy_bio35_usd_41,lh_levy_bio35_usd_44,lh_levy_bio35_usd_47,lh_levy_bio35_usd_50,asis_bio35_usd_26,asis_bio35_usd_29,asis_bio35_usd_32,asis_bio35_usd_35,asis_bio35_usd_38,asis_bio35_usd_41,asis_bio35_usd_44,asis_bio35_usd_47,asis_bio35_usd_50,brazil_bio35_usd_26,brazil_bio35_usd_29,brazil_bio35_usd_32,brazil_bio35_usd_35,brazil_bio35_usd_38,brazil_bio35_usd_41,brazil_bio35_usd_44,brazil_bio35_usd_47,brazil_bio35_usd_50,liscr_bio35_usd_26,liscr_bio35_usd_29,liscr_bio35_usd_32,liscr_bio35_usd_35,liscr_bio35_usd_38,liscr_bio35_usd_41,liscr_bio35_usd_44,liscr_bio35_usd_47,liscr_bio35_usd_50,liscr_tolng_bio35_usd_26,liscr_tolng_bio35_usd_29,liscr_tolng_bio35_usd_32,liscr_tolng_bio35_usd_35,liscr_tolng_bio35_usd_38,liscr_tolng_bio35_usd_41,liscr_tolng_bio35_usd_44,liscr_tolng_bio35_usd_47,lis

## 3.3 Evaluate NZF Costs

In [12]:
years=["26", "29", "32", "35", "38", "41", "44", "47", "50"]
hs_codes_not_mapped_to_HS, vess_types_not_found_in_inventories = [], []

for year in years:

    # Setup columns for policy costs per bilateral goods trade.
    bau_usd, levy_usd, lh_levy_usd, asis_usd, brazil_usd, liscr_usd = [], [], [], [], [], []
    
    for trade_idx, trade in trade_ex.iterrows():

        # If no facilitating vessel types were picked up for the HS code, take from full voyages dataset, otherwise move on...
        if 'all' in trade.fac_vessel_types:
            hs_codes_not_mapped_to_HS.append(trade.desc)
            int_dep_voys_vess = voys_fem

        else:
            # 1. If possible, derive subset of voyages associated with the unique vessel type.
            int_dep_voys_vess = int_dep_voys[(int_dep_voys.vessel_type.isin(trade.fac_vessel_types))]
    
            # 2. If no voyages can be found, take from other international voyages dataset instead.
            if int_dep_voys_vess.shape[0] == 0:
                int_dep_voys_vess = int_arr_voys[(int_arr_voys.vessel_type.isin(trade.fac_vessel_types))]

            # 3. Finally, if no voyages are found as the vessel type doesn't appear in the country's shipping inventories, also take from full voyages dataset filtered for those vessel types instead.
            if int_dep_voys_vess.shape[0] == 0:
                [vess_types_not_found_in_inventories.append(trade.fac_vessel_types[i]) for i in range(len(trade.fac_vessel_types))]
                int_dep_voys_vess = voys_fem[(voys_fem.vessel_type.isin(trade.fac_vessel_types))]

            if int_dep_voys_vess.shape[0] == 0:
                print("Trade unnaccounted for: \n\n{0}\n".format(trade))
    
        # Add cost intensities associated with the subset of voyages by vessel type.
        bau_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "bau_bio35_usd_per_dwtnm_"+year))
        levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "levy_bio35_usd_per_dwtnm_"+year))
        lh_levy_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "lh_levy_bio35_usd_per_dwtnm_"+year))
        asis_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "asis_bio35_usd_per_dwtnm_"+year))
        brazil_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "brazil_bio35_usd_per_dwtnm_"+year))
        liscr_usd.append(trade.tnm * get_usd_per_tnm(int_dep_voys_vess, "liscr_bio35_usd_per_dwtnm_"+year))

    trade_ex["bau_usd_"+year] = bau_usd
    trade_ex["levy_usd_"+year] = levy_usd
    trade_ex["lh_levy_usd_"+year] = lh_levy_usd
    trade_ex["asis_usd_"+year] = asis_usd
    trade_ex["brazil_usd_"+year] = brazil_usd
    trade_ex["liscr_usd_"+year] = liscr_usd

print("Full voyages table used for the following commodities as no vessel types could be found: \n\n{0}\n".format(pd.Series(hs_codes_not_mapped_to_HS).unique()))
print("Full voyages table filtered for these vessel types as they couldn't be found in the inventories: \n\n{0}\n".format(pd.Series(vess_types_not_found_in_inventories).unique()))

print()
print("Number of trade records present in dataset: {0}".format(trade_ex.shape[0]))
print("Number of partner countries present in dataset: {0}".format(len(trade_ex.partnerCode.unique())))
print("\nTotal value of trade: US${0}m".format(np.round(trade_ex.fobvalue.sum() / 1e6, 1)))
print("Total weight within the dataset: {0} million kilograms".format(np.round(trade_ex.netWgt.sum() / 1e6, 1)))

print("\n\nTotal FEM costs in 2041...\n")
print("BAU: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.bau_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.bau_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Levy (low-then-high): US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.lh_levy_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.lh_levy_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("'as-is': US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.asis_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.asis_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("Brazil: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.brazil_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.brazil_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))
print("LISCR: US${0}m ({1}% ad-val)".format(
    np.round(trade_ex.liscr_usd_41.sum() / 1e6, 1),
    round(100 * trade_ex.liscr_usd_41.sum() / trade_ex.fobvalue.sum(), 1)
))

trade_ex_na = trade_ex[(trade_ex[["netWgt", "primaryValue", "HS2", "cmdCode", "sea_dist_nm", "tnm", "bau_usd_26"]].isnull().any(axis=1)) | (trade_ex.fac_vessel_types.str.len() == 0)]
print("Rows with NA: ${0} ({1}%)\n\n".format(
    trade_ex_na.primaryValue.sum(), 
    round(100 * trade_ex_na.primaryValue.sum() / trade_ex.primaryValue.sum(), 1)
))

trade_ex.to_csv(output_dir + "{0}/impact_ex_{1}.csv".format(iso_country, version), index=False)
trade_ex.head(2)

Full voyages table used for the following commodities as no vessel types could be found: 

['3827 | Mixtures containing ' '8807 | Aircraft; parts of g'
 '9620 | Monopods, bipods, tr']

Full voyages table filtered for these vessel types as they couldn't be found in the inventories: 

[]


Number of trade records present in dataset: 3552
Number of partner countries present in dataset: 114

Total value of trade: US$2931.7m
Total weight within the dataset: 1866.6 million kilograms


Total FEM costs in 2041...

BAU: US$62.7m (2.1% ad-val)
Levy: US$131.3m (4.5% ad-val)
Levy (low-then-high): US$129.2m (4.4% ad-val)
'as-is': US$123.5m (4.2% ad-val)
Brazil: US$127.1m (4.3% ad-val)
LISCR: US$88.2m (3.0% ad-val)
Rows with NA: $0.0 (0.0%)




,trade_idx,reporterCode,reporterDesc,partnerCode,partnerDesc,partner2Code,partner2Desc,flowDesc,motDesc,cmdDesc,cmdCode,qtyUnitAbbr,qty,netWgt,cifvalue,fobvalue,primaryValue,desc_full,desc,bilat_desc_full,HS2,HS2_desc,HS2_desc_full,bilat_HS2_desc_full,fac_vessel_types,sea_dist_km,sea_dist_nm,tnm,bau_usd_26,levy_usd_26,lh_levy_usd_26,asis_usd_26,brazil_usd_26,liscr_usd_26,bau_usd_29,levy_usd_29,lh_levy_usd_29,asis_usd_29,brazil_usd_29,liscr_usd_29,bau_usd_32,levy_usd_32,lh_levy_usd_32,asis_usd_32,brazil_usd_32,liscr_usd_32,bau_usd_35,levy_usd_35,lh_levy_usd_35,asis_usd_35,brazil_usd_35,liscr_usd_35,bau_usd_38,levy_usd_38,lh_levy_usd_38,asis_usd_38,brazil_usd_38,liscr_usd_38,bau_usd_41,levy_usd_41,lh_levy_usd_41,asis_usd_41,brazil_usd_41,liscr_usd_41,bau_usd_44,levy_usd_44,lh_levy_usd_44,asis_usd_44,brazil_usd_44,liscr_usd_44,bau_usd_47,levy_usd_47,lh_levy_usd_47,asis_usd_47,brazil_usd_47,liscr_usd_47,bau_usd_50,levy_usd_50,lh_levy_usd_50,asis_usd_50,brazil_usd_50,liscr_usd_50
0,0,516,Namibia,056,Belgium,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,72363.0,72363.0,772543.778,772543.778,772543.778,Meat of bovine anima,0201 | Meat of bovine anima,Belgium | 0201 | Meat of bovine anima,02,02 | Meat and edible meat,Meat and edible meat,Belgium | 02 | Meat and edible meat,[Container],10468.0,6504.511628,470685.974937,3185.406869,3171.784098,3172.231749,3172.680425,3172.680425,3181.923065,3347.706053,6387.656293,3901.816742,3743.784461,3391.189036,3405.047140,3270.528674,6029.679126,4277.705784,3859.382959,3539.256843,3463.283592,3202.931096,6644.584215,6599.521829,4850.200166,4206.261054,3893.401477,3136.152171,7071.645283,6536.194848,5619.679997,5294.03926,4117.158975,3070.585377,7173.655451,7017.296104,6665.870424,6766.416554,4501.701203,3003.782402,7060.397857,7153.977935,6852.803656,7228.296263,5280.374188,2940.974207,6305.207624,6617.620673,6410.880290,6718.580482,5114.702996,2877.173346,5628.531640,5744.695728,5795.587222,6833.240721,5573.165901
1,1,516,Namibia,180,Dem. Rep. of the Congo,000,World,Export,Sea,Meat of bovine animals; fresh or chilled,0201,kg,55.0,55.0,217.547,217.547,217.547,Meat of bovine anima,0201 | Meat of bovine anima,Dem. Rep. of the Congo | 0201 | Meat of bovine...,02,02 | Meat and edible meat,Meat and edible meat,Dem. Rep. of the Congo | 02 | Meat and edible ...,[Container],2222.0,1380.686362,75.937750,0.513915,0.511717,0.511790,0.511862,0.511862,0.513353,0.540100,1.030547,0.629497,0.604001,0.547115,0.549351,0.527648,0.972794,0.690140,0.622651,0.571003,0.558746,0.516742,1.071999,1.064729,0.782503,0.678614,0.628139,0.505969,1.140898,1.054512,0.906647,0.85411,0.664239,0.495390,1.157356,1.132130,1.075433,1.091654,0.726278,0.484613,1.139084,1.154181,1.105592,1.166171,0.851905,0.474480,1.017246,1.067649,1.034294,1.083937,0.825177,0.464186,0.908075,0.926816,0.935026,1.102435,0.899142


## 3.4 Inspect Output

### 3.4.1 Total Costs per Scenario

In [13]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        np.round(trade_ex["bau_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["lh_levy_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["asis_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["brazil_usd_{0}".format(year)].sum() / 1e6, 1),
        np.round(trade_ex["liscr_usd_{0}".format(year)].sum() / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["BAU", "Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * trade_ex["bau_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["lh_levy_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["asis_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["brazil_usd_{0}".format(year)].sum() / gdp, 3),
        round(100 * trade_ex["liscr_usd_{0}".format(year)].sum() / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_total_costs_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,67.0,65.8,64.7,63.7,62.7,61.7,60.7,59.8
1,Levy,112.0,107.2,117.5,128.0,131.3,124.6,114.1,105.8
2,Levy (LH),74.8,80.9,119.7,123.9,129.2,126.9,119.2,106.9
3,'as-is',71.9,74.2,89.8,102.6,123.5,121.9,116.6,107.7
4,Brazil,67.1,69.9,81.4,97.4,127.1,128.5,124.9,124.2
5,LISCR,67.3,68.8,76.5,80.2,88.2,102.0,99.6,103.9



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,2.3,2.2,2.2,2.2,2.1,2.1,2.1,2.0
1,Levy,3.8,3.7,4.0,4.4,4.5,4.3,3.9,3.6
2,Levy (LH),2.6,2.8,4.1,4.2,4.4,4.3,4.1,3.6
3,'as-is',2.5,2.5,3.1,3.5,4.2,4.2,4.0,3.7
4,Brazil,2.3,2.4,2.8,3.3,4.3,4.4,4.3,4.2
5,LISCR,2.3,2.3,2.6,2.7,3.0,3.5,3.4,3.5



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,BAU,0.497,0.489,0.481,0.473,0.466,0.458,0.451,0.444
1,Levy,0.832,0.796,0.873,0.951,0.975,0.926,0.848,0.786
2,Levy (LH),0.556,0.601,0.889,0.921,0.960,0.943,0.886,0.794
3,'as-is',0.534,0.551,0.667,0.762,0.918,0.905,0.867,0.800
4,Brazil,0.498,0.520,0.604,0.723,0.945,0.955,0.928,0.922
5,LISCR,0.500,0.511,0.568,0.595,0.656,0.757,0.740,0.772


### 3.4.2 Costs per Scenario relative to BAU

In [14]:
print("\nTotal FEM costs by scenario (US$m)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    
    d_temp = {"20{0}".format(year): [
        np.round((trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1),
        np.round((trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / 1e6, 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_usd_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% ad-val)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / trade_ex.fobvalue.sum(), 1)
    ]}
    d = d | d_temp
df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_adval_{1}.csv".format(iso_country, version), index=False)
display(df)

print("\nTotal FEM costs by scenario (% gdp)...\n")
d = {"Scenario":["Levy", "Levy (LH)", "'as-is'", "Brazil", "LISCR"]}
for year in ["29", "32", "35", "38", "41", "44", "47", "50"]:
    d_temp = {"20{0}".format(year): [
        round(100 * (trade_ex["levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["lh_levy_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["asis_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["brazil_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3),
        round(100 * (trade_ex["liscr_usd_{0}".format(year)].sum() - trade_ex["bau_usd_{0}".format(year)].sum()) / gdp, 3)
    ]}
    d = d | d_temp

df = pd.DataFrame(data=d)
df.to_csv(output_dir + "{0}/impact_ex_costs_to_bau_pct_gdp_{1}.csv".format(iso_country, version), index=False)
display(df)


Total FEM costs by scenario (US$m)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,45.0,41.4,52.7,64.3,68.5,62.9,53.4,46.0
1,Levy (LH),7.9,15.2,54.9,60.2,66.5,65.2,58.5,47.1
2,'as-is',4.9,8.4,25.1,38.9,60.8,60.2,55.9,48.0
3,Brazil,0.1,4.2,16.6,33.6,64.4,66.8,64.1,64.4
4,LISCR,0.3,3.1,11.8,16.4,25.5,40.3,38.8,44.1



Total FEM costs by scenario (% ad-val)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,1.5,1.4,1.8,2.2,2.3,2.1,1.8,1.6
1,Levy (LH),0.3,0.5,1.9,2.1,2.3,2.2,2.0,1.6
2,'as-is',0.2,0.3,0.9,1.3,2.1,2.1,1.9,1.6
3,Brazil,0.0,0.1,0.6,1.1,2.2,2.3,2.2,2.2
4,LISCR,0.0,0.1,0.4,0.6,0.9,1.4,1.3,1.5



Total FEM costs by scenario (% gdp)...



,Scenario,2029,2032,2035,2038,2041,2044,2047,2050
0,Levy,0.334,0.308,0.392,0.478,0.509,0.468,0.397,0.342
1,Levy (LH),0.059,0.113,0.408,0.447,0.494,0.484,0.435,0.350
2,'as-is',0.037,0.063,0.186,0.289,0.452,0.447,0.415,0.356
3,Brazil,0.001,0.031,0.124,0.250,0.479,0.496,0.477,0.479
4,LISCR,0.003,0.023,0.087,0.122,0.190,0.299,0.289,0.328
